[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Series &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


**1.** Make a Series from the dictionary `LATITUDES`, and print the name of the northernmost station with `idxmax`.

In [2]:
latitude = pd.Series(LATITUDES)

print(latitude.idxmax(), latitude.max())

Svalbard 78.22


`idxmax` returns the label of the largest value. Svalbard, at 78.22 degrees north, is the northernmost.

**2.** Convert `planned` to Fahrenheit, and print only the stations whose mean is above 40 °F.

In [3]:
planned = pd.Series(STATIONS)
fahrenheit = planned * 9 / 5 + 32

print(fahrenheit[fahrenheit > 40])

Bergen    46.4
Oslo      43.7
dtype: float64


The first line builds `planned` as the worked example did, since this notebook runs on its own. Bergen and Oslo are above 40 °F. Tromso's 3.5 °C is 38.3 °F.

**3.** Make Svalbard's readings a Series indexed by time, the way the last worked example made Oslo's, and print its reading at 15:00 on 13 July.

In [4]:
svalbard = readings[readings["station"] == "Svalbard"]
svalbard_by_time = pd.Series(svalbard["celsius"].to_numpy(), index=svalbard["time"].to_numpy(), name="Svalbard")

print(svalbard_by_time.loc["2025-07-13 15:00"])

8.3


Filtering once and taking both columns from the result keeps the values and the times in step. 8.3 is Svalbard's warmest reading of the year, as the **Why Pandas** notebook found.

**4.** Print the labels of Svalbard's first three readings in `readings["celsius"]` after filtering on the station, and say why they are not 0, 1 and 2.

In [5]:
svalbard_celsius = readings[readings["station"] == "Svalbard"]["celsius"]

print(svalbard_celsius.index[:3].tolist())

[2, 6, 10]


The labels are 2, 6 and 10. Svalbard is the third station each hour, so its readings sit at rows 2, 6 and 10 of the table, and filtering keeps those rows' labels.

**5.** Index Tromso's and Oslo's readings by time, subtract Oslo's from Tromso's hour by hour, and print the mean of the difference rounded to two places.

In [6]:
def by_time(station):
    rows = readings[readings["station"] == station]
    return pd.Series(rows["celsius"].to_numpy(), index=rows["time"].to_numpy(), name=station)


print(round((by_time("Tromso") - by_time("Oslo")).mean(), 2))

-3.0


A small function builds each station's Series the same way. Tromso is 3 degrees colder than Oslo on average, the difference between the formula's means of 3.5 and 6.5.

---

&#8592; **Back to:** [Series](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/02-series.ipynb)  &nbsp;&middot;&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)
